### Notebook 范围

### 目的
分析 Jan/Feb 可用窗口中，Z300 stationary-wave geometry 是否解释 EP100 anomaly。

### 科学问题
对流层 300 hPa stationary-wave 图形、wave2 amplitude 或与参考年的 ACC，是否对应 lower-stratospheric EP100 W1+W2 anomaly？

### 方法
只比较 Jan 和 Feb calendar windows；非一月/二月已初始化前缺失的窗口记为 NA。Z300 anomaly 去除 zonal mean；climatology target 用 B2000 同月 300 hPa stationary waves；ACC_to_reference 是成员 Z300 stationary anomaly 与同年 BWCN reference Z300 stationary anomaly 的 pattern correlation。EP100 anomaly = hindcast member EP100 - BWCN same-year EP100 reference。

### 输出
outputs/figures/05_z300_source 和 outputs/tables/05_z300_source。

### 解读
如果 projection/closeness/ACC/wave2 amplitude 与 EP100 anomaly 高相关，说明 tropospheric planetary-wave geometry 可能是 EP100 spread 来源。

### 注意
BWCN reference EP100 目前来自 omega-corrected product，而 hindcast EP100 使用 no-omega-consistent EPflux_daily_ubar；因此 anomaly 是近似 dynamical-error 指标。


### 导入与公共路径

### 目的
加载 Extention_analysis 公共函数。

### 科学问题
所有 notebook 是否共享相同路径、变量定义和 sign convention？

### 方法
导入 hindcast_extension_utils.py。

### 输出
无图。

### 解读
路径正确即可继续。

### 注意
所有输出都写入 outputs 子目录。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 计算 Jan/Feb Z300 source metrics

### 目的
按 case-month 计算 Z300 projection、closeness、ACC、wave amplitude，并与 EP100 anomaly 相关。

### 科学问题
ACC 应该对照 reference year；它是否能解释成员相对参考年的 EP100 偏差？

### 方法
window 只取 Jan 和 Feb。Z300 wave-k amplitude 是去除 zonal mean 后，对 longitude 做 Fourier k 波幅，在 20-90N cos-lat 平均，单位米。

### 输出
05_Z300_source_metric_heatmap_JanFeb_v2.png/pdf 和 CSV。

### 解读
ACC_to_reference 高且 EP100 anomaly 小/大之间有相关，说明越接近/偏离 reference tropospheric pattern 可能影响波进入平流层。

### 注意
Jan/Feb 以外的 March/April 初始化 case 不用于这个 source-window 解释；它们可用于 feedback/FWD，而非 Jan/Feb source diagnosis。


In [ ]:
fig_dir = figure_dir("05_z300_source")
tab_dir = table_dir("05_z300_source")
inv = discover_hindcast_cases()
rows = []
for _, meta_row in inv.iterrows():
    case = meta_row["case"]; year = meta_row["year"]
    for month in ["Jan", "Feb"]:
        if not case_month_window_available(case, month):
            continue
        window = MONTH_WINDOWS[month]
        z = load_or_build_z300(case, window)
        target = compute_z300_climatological_stationary_target(month)
        ref_z = load_or_build_bwcn_z300(year, window)
        ep = compute_all_ep100(case, window)
        if z is None or ep.empty:
            log_message(f"{case} {month}: skip Z300 metrics, missing Z300 or EP100")
            continue
        ref_za = compute_z300_stationary_anomaly(ref_z) if ref_z is not None else None
        ref_ep_w1 = load_bwcn_ep100_reference(year, "wave1", window)
        ref_ep_w2 = load_bwcn_ep100_reference(year, "wave2", window)
        ref_ep_rest = load_bwcn_ep100_reference(year, "wave_rest", window)
        ref_ep_all = load_bwcn_ep100_reference(year, "all_waves", window)
        ref_ep_w1w2 = ref_ep_w1 + ref_ep_w2 if np.isfinite(ref_ep_w1) and np.isfinite(ref_ep_w2) else np.nan
        ep = ep.copy()
        ep["EP100_wave1_plus_wave2_anom_ref"] = ep["EP100_wave1_plus_wave2"] - ref_ep_w1w2
        ep["EP100_wave2_anom_ref"] = ep["EP100_wave2"] - ref_ep_w2
        ep["EP100_all_waves_anom_ref"] = ep["EP100_all_waves"] - ref_ep_all
        ep_index = ep.set_index("member")
        for mid in z["member"].values:
            member = member_short_id(mid)
            if member not in ep_index.index:
                continue
            za = compute_z300_stationary_anomaly(z.sel(member=mid))
            row = {"case": case, "year": year, "month": month, "member": member, "window": window_to_label(window)}
            row["Z300_stationary_projection_to_B2000"] = weighted_projection(za, target)
            row["Z300_closeness_to_B2000"] = weighted_pattern_corr(za, target)
            row["Z300_ACC_to_BWCN_reference"] = weighted_pattern_corr(za, ref_za)
            for k in [1, 2, 3, 4, 5, 6]:
                row[f"Z300_wave{k}_amplitude_m"] = z300_wave_amplitude(za, k)
            row["Z300_synoptic_3_6_amplitude_m"] = math.sqrt(sum(row[f"Z300_wave{k}_amplitude_m"]**2 for k in [3,4,5,6]))
            for col in ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave_rest", "EP100_wave1_plus_wave2", "EP100_all_waves_anom_ref", "EP100_wave2_anom_ref", "EP100_wave1_plus_wave2_anom_ref"]:
                row[col] = float(ep_index.loc[member, col]) if col in ep_index else np.nan
            rows.append(row)
metrics = pd.DataFrame(rows)
metrics_csv = tab_dir / "05_Z300_source_metrics_JanFeb.csv"
metrics.to_csv(metrics_csv, index=False)
# Root copy for synthesis.
metrics.to_csv(TAB_DIR / "05_Z300_source_metrics_all_cases.csv", index=False)
relationships = [
    ("Z300_stationary_projection_to_B2000", "EP100_wave1_plus_wave2_anom_ref"),
    ("Z300_closeness_to_B2000", "EP100_wave1_plus_wave2_anom_ref"),
    ("Z300_ACC_to_BWCN_reference", "EP100_wave1_plus_wave2_anom_ref"),
    ("Z300_wave2_amplitude_m", "EP100_wave2_anom_ref"),
    ("Z300_synoptic_3_6_amplitude_m", "EP100_all_waves_anom_ref"),
]
cor_rows = []
for (case, month), sub in metrics.groupby(["case", "month"]) if not metrics.empty else []:
    for xcol, ycol in relationships:
        cor_rows.append({"case": case, "month": month, "relationship": f"{xcol} vs {ycol}", **finite_corr(sub[xcol], sub[ycol])})
cor = pd.DataFrame(cor_rows)
cor_csv = tab_dir / "05_Z300_source_metric_correlations_JanFeb.csv"
cor.to_csv(cor_csv, index=False)
cor.to_csv(TAB_DIR / "05_Z300_source_metric_correlations_all_cases.csv", index=False)
mat = cor.pivot(index=["case", "month"], columns="relationship", values="R") if not cor.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(14.5, max(5, 0.34 * len(mat) + 2)), constrained_layout=True)
if mat.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No Jan/Feb Z300 source metrics", ha="center")
else:
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(mat.shape[1]), [c.replace("Z300_", "").replace("EP100_", "EP ").replace("_", " ") for c in mat.columns], rotation=45, ha="right")
    ax.set_yticks(range(mat.shape[0]), [f"{i[0]} {i[1]}" for i in mat.index])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title("Z300 source metrics vs EP100 reference anomaly (Jan/Feb only)\nACC = corr(member Z300 stationary anomaly, BWCN same-year reference Z300); EP100 = mean(-ep2), 100 hPa, 40-80N")
    fig.colorbar(im, ax=ax, label="Pearson R")
savefig(fig, "05_Z300_source_metric_heatmap_JanFeb_v2", fig_dir=fig_dir, notebook="05_Z300_stationary_source_multicase.ipynb", scientific_question="Jan/Feb Z300 stationary-wave geometry 是否解释 EP100 reference anomaly？", variables_windows="Z300 300 hPa 20-90N; B2000 Jan/Feb stationary target; BWCN same-year Z300 reference; EP100 anomaly at 100 hPa 40-80N", interpretation="高 |R| 表示 tropospheric stationary-wave geometry 与 lower-stratospheric wave forcing error 有联系。", caveat="BWCN EP100 reference 是 omega-corrected；hindcast EP100 为 no-omega-consistent 产品。", csv_table=cor_csv)
plt.show()


### 绘制 Jan/Feb pointwise correlation maps

### 目的
为每个可用 case-month 绘制 Z300 gridpoint correlation maps。

### 科学问题
二维 Z300 anomaly pattern 哪些区域与 EP100 anomaly 或 O3 RMSE 相关？

### 方法
颜色是 gridpoint member correlation；黑色等值线是 B2000 同月 300 hPa climatological stationary waves。若 cartopy 可用，叠加海岸线。

### 输出
05_Z300_pointwise_corr_<case>_<month>_v2.png/pdf 和逐点 summary CSV。

### 解读
相关正负中心若贴近 climatological ridge/trough，可支持 constructive/destructive interference 解释。

### 注意
pointwise p 值未做多重检验校正；这张图是 source diagnosis 的探索图。


In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False
    ccrs = None
    cfeature = None

def _draw_corr_panel(ax, ds, target, title):
    if HAS_CARTOPY:
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.25)
        transform = ccrs.PlateCarree()
    else:
        transform = None
    im = ax.contourf(ds["lon"], ds["lat"], ds["R"], levels=np.linspace(-0.8, 0.8, 17), cmap="RdBu_r", extend="both", transform=transform)
    if target is not None:
        ax.contour(target["lon"], target["lat"], target, levels=np.arange(-240, 241, 40), colors="k", linewidths=0.45, transform=transform)
    ax.set_title(title, fontsize=9)
    return im

summary_rows = []
for (case, month), subm in metrics.groupby(["case", "month"]) if not metrics.empty else []:
    window = MONTH_WINDOWS[month]
    z = load_or_build_z300(case, window)
    if z is None:
        continue
    za = compute_z300_stationary_anomaly(z)
    target = compute_z300_climatological_stationary_target(month)
    ep = subm.set_index("member")
    o3, _ = load_hindcast_o3(case)
    ref_o3, _ = load_bwcn_reference_o3(parse_case_name(case)["year"])
    rmse = compute_o3_rmse(o3, ref_o3, *target_window_for_case(case)).set_index("member")
    metrics_for_map = ep[["EP100_all_waves_anom_ref", "EP100_wave1_plus_wave2_anom_ref", "EP100_wave2_anom_ref"]].join(rmse[["O3_RMSE"]], how="left")
    targets = ["EP100_all_waves_anom_ref", "EP100_wave1_plus_wave2_anom_ref", "EP100_wave2_anom_ref", "O3_RMSE"]
    subplot_kw = {"projection": ccrs.PlateCarree()} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), subplot_kw=subplot_kw, constrained_layout=True)
    im = None
    case_rows = []
    for ax, col in zip(axes.ravel(), targets):
        ds = pointwise_member_correlation(za, metrics_for_map[col])
        if ds is None:
            ax.axis("off"); ax.set_title(f"{col}: missing")
            continue
        im = _draw_corr_panel(ax, ds, target, col)
        case_rows.append({"case": case, "month": month, "target": col, "R_mean": float(ds["R"].mean(skipna=True)), "R_absmax": float(abs(ds["R"]).max(skipna=True))})
    if im is not None:
        fig.colorbar(im, ax=axes.ravel().tolist(), orientation="horizontal", fraction=0.045, pad=0.05, label="member correlation R")
    fig.suptitle(f"{case} {month}: Z300 stationary anomaly pointwise correlations\nContours: B2000 {month} climatological stationary waves at 300 hPa")
    case_csv = tab_dir / f"05_Z300_pointwise_corr_{case}_{month}_summary.csv"
    pd.DataFrame(case_rows).to_csv(case_csv, index=False)
    summary_rows.extend(case_rows)
    savefig(fig, f"05_Z300_pointwise_corr_{case}_{month}_v2", fig_dir=fig_dir / "pointwise_maps" / case[:4], notebook="05_Z300_stationary_source_multicase.ipynb", scientific_question="哪些 Z300 anomaly 区域与 EP100/O3 指标相关？", variables_windows=f"{case} {month}; Z300 300 hPa stationary anomaly; EP100 reference anomaly; O3 RMSE", interpretation="相关中心若与 stationary-wave contours 对齐，支持 wave interference source 解释。", caveat="探索性点相关，未做多重检验校正。", csv_table=case_csv)
    plt.close(fig)
summary = pd.DataFrame(summary_rows)
summary.to_csv(tab_dir / "05_Z300_pointwise_corr_selected_summary.csv", index=False)
summary.to_csv(TAB_DIR / "05_Z300_pointwise_corr_selected_summary.csv", index=False)
write_figure_guide()
